In [13]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus
/home/yuanshanwu/Documents/GitHub/QuantUS/engines


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [14]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['mp4', 'custom_dicom', 'nifti', 'avi']


In [15]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [16]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [17]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['load_bolus_mask', 'nifti']


In [18]:
seg_type = 'nifti'

seg_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/MotionCompensated_QUANTUSGUI.nii.gz'
seg_loader_kwargs = {}

In [20]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Load data

In [5]:
import pickle
with open('/home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/First_MC_analysis/seg_data_with_mc_correlation_based.pkl', 'rb') as f:
    seg_data = pickle.load(f)

# CEUS Quantitative Temporal Curve Visualization

In [7]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))
print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis types: ['curves', 'curves_paramap']
Available analysis functions: ['pyradiomics', 'tic']


In [8]:
analysis_type = 'curves_paramap'
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['cor_vox_ovrlp', 'cor_vox_len', 'ax_vox_ovrlp', 'sag_vox_len', 'sag_vox_ovrlp', 'ax_vox_len']


In [9]:
analysis_kwargs = {
    'ax_vox_ovrlp': 70,  # %
    'sag_vox_ovrlp': 70,  # %
    'cor_vox_ovrlp': 70,  # %
    'ax_vox_len': 5.0,  # mm
    'sag_vox_len': 2.5,  # mm
    'cor_vox_len': 2.5,  # mm
    'curves_output_path': '', # don't export the curves we generate
}

In [13]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 530/530 [6:11:22<00:00, 42.04s/it]  


In [14]:
# Save the segmentation data with motion compensation applied
import pickle
from pathlib import Path

# Build output path: split CEUS_scan_path at 'HighQuality' and append pkl filename
base_path = CEUS_scan_path.split('HighQuality')[0]
output_pkl_path = str(Path(base_path) / 'HighQuality' / 'ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl')
print(f"Saving to: {output_pkl_path}")
# Save seg_data with motion compensation
with open(output_pkl_path, 'wb') as f:
    pickle.dump(analysis_obj, f)

[autoreload of cv2.gapi failed: Traceback (most recent call last):
  File "/home/yuanshanwu/Documents/GitHub/QuantUS/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/yuanshanwu/Documents/GitHub/QuantUS/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/home/yuanshanwu/Documents/GitHub/QuantUS/.venv/lib/python3.12/site-packages/cv2/gapi/__init__.py", line 323, in <module>
    cv.gapi.wip.GStreamerPipeline = cv.gapi_wip_gst_GStreamerPipeline
    ^^^^^^^^^^^
AttributeErro

Saving to: /home/yuanshanwu/Documents/TUL/CEUS-Studies/P07-V02-CE01/HighQuality/ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl


## Curve Quantification for each sub-kernels

In [6]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [16]:
function_names = ['lognormal_fit_full'] # Empty list will use all functions
output_path = 'test_quants.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
}

In [17]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

## Visualize the parametric Map

In [18]:
from src.visualizations.options import get_visualization_types

types, funcs = get_visualization_types()
print("Available visualization types:", list(types.keys()))
print("Available visualization functions:", list(funcs.keys()))

Available visualization types: ['paramap']
Available visualization functions: ['paramap']


In [19]:
vis_type = 'paramap'
params = [] # Empty list will use all parameters 
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': 'paramaps_LOGNORMAL',
    'hide_all_visualizations': False,  # Set to True to hide all visualizations
}

In [20]:
from src.entrypoints import visualization_step

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)

In [24]:
vis_obj.paramap_names

['AUC_full_TIC',
 'PE_full_TIC',
 'TP_full_TIC',
 'MTT_full_TIC',
 'T0_full_TIC',
 'Mu_full_TIC',
 'Sigma_full_TIC',
 'PE_Ix_full_TIC']

In [26]:
from src.map_showing import *

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_T0_with_CEUS(vis_obj, image_data, 
                          param_name='PE_full_TIC',
                          time_point=0)  # Frame 50
napari.run()

In [ ]:
vis_obj.paramap_names

# Pixel-wise T0 map

In [7]:
seg_data.use_mc = False
seg_data.seg_mask.shape

(195, 144, 145)

In [8]:
from src.pixel_T0 import generate_t0_map_3d, get_t0_statistics_3d

t0_map = generate_t0_map_3d(
    image_data, seg_data,
    threshold=130, start_frame=0, end_frame=150,
    min_consecutive_frames=3
)
stats = get_t0_statistics_3d(t0_map, seg_data)

In [9]:
print("Non Motion Compensated T0 Map Statistics:")
print(stats)

Non Motion Compensated T0 Map Statistics:
{'mean_t0': 78.42781829833984, 'median_t0': 80.0, 'std_t0': 25.97692108154297, 'min_t0': 4.0, 'max_t0': 138.0, 'coverage': 16.48424230986979}


In [10]:
from src.map_showing import *

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_heatmap(t0_map, image_data, 
                      time_point=0, seg_mask=seg_data.seg_mask)  #0
napari.run()

T0 Map: contrast=[0, 123.0], mean(activated)=78.4, activated voxels=8735


In [11]:
seg_data.use_mc = True
t0_map_mc = generate_t0_map_3d(
    image_data, seg_data,
    threshold=130, start_frame=0, end_frame=150,
    min_consecutive_frames=3
)
stats_mc = get_t0_statistics_3d(t0_map_mc, seg_data)
print("Motion Compensated T0 Map Statistics:")
print(stats_mc)

Motion Compensated T0 Map Statistics:
{'mean_t0': 76.7054214477539, 'median_t0': 80.0, 'std_t0': 26.783710479736328, 'min_t0': 4.0, 'max_t0': 134.0, 'coverage': 9.33383657293829}


In [12]:
viewer = view_heatmap(t0_map_mc, image_data, 
                      time_point=0, seg_mask=seg_data.seg_mask)  #0
napari.run()

T0 Map: contrast=[0, 119.0], mean(activated)=75.0, activated voxels=6021
